In [ ]:
#Set-up the plot
fig, axes = plt.subplots(
    som_y, som_x,
    figsize=(18, 13),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True
)
fig.suptitle('Environmental Feature Input SOM', fontsize=24)

#Function to relate SOM standardized/trained data to original data for plotting
for j in range(som_y):
    for i in range(som_x):

        ax = axes[j, i]

        node_mask = (bmus[:, 0] == i) & (bmus[:, 1] == j)
        n_cases = node_mask.sum()

        if n_cases == 0:
            ax.set_title(f"Node {j+1},{i+1}\nNo cases")
            ax.axis("off")
            continue

        #Calling original data instead of standardized data
        cape_node = ds["cape"].isel(time=node_mask).mean("time")
        cin_node  = ds["cin"].isel(time=node_mask).mean("time")

        #Calling the geography from original dataset
        lon = ds["lon"]
        lat = ds["lat"]

        #Visualization of CAPE on each node
        pm = ax.pcolormesh(
            lon, lat, cape_node,
            transform=ccrs.PlateCarree(),
            shading="auto",
            cmap='Pivotal_CAPE'
        )

        #Hatching for CIN on each node
        ax.contourf(
            lon, lat, cin_node,
            levels=[-1e9, -75],
            colors="none",
            hatches=["////"],
            transform=ccrs.PlateCarree()
        )
        ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree()) #Plotting to just CONUS
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.STATES, linewidth=0.3)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)

        ax.set_title(f"Node {j+1},{i+1}\n n={n_cases}")

#Create a colorbar
fig.colorbar(pm, ax=axes.ravel().tolist(), shrink=0.8,orientation='horizontal', label="Mean CAPE")

#Save figure as .png
plt.savefig('SOM_Out.png', dpi=300)

#display plot in notebook
plt.show()